# B07-00 Clean v2 | 배지 조성에 근거한 ML 데이터 감사

실제 Round05 자료의 `Cond00577` (`K2HPO4`: 30.75/61.50 mM)과 `Cond00578` (`Tryptophan`: 0.20/2.00 mM)에는 하나의 Condition ID 아래 서로 다른 조성이 존재한다. **이 차이는 실재하는 원본 기록이며 단순 반올림 오차로 취급하지 않는다.**

v2에서는 원본 ID를 보존하고, 배지 성분 전체를 기반으로 결정적 `medium_profile_id`를 생성한다. 같은 조성은 Condition ID가 달라도 같은 profile ID로 묶는다. 머신러닝의 그룹 분할은 이 profile ID를 이용한다. 원본 값은 변경하지 않으며, 이 방식은 원 실험의 설계상 의도를 새로 증명하지 않는다.

본 Notebook은 아직 ML을 학습하지 않으며, 원본 `K`, `r`의 생물학적 정의·단위를 확정하지 않는다.

## 1. 입력 경로와 승인 게이트

In [ ]:
from pathlib import Path
import json, hashlib, re, sys
import numpy as np
import pandas as pd
cwd=Path.cwd().resolve()
ROOT=next((x for x in (cwd,cwd.parent,cwd/'bioimage-ai') if (x/'data'/'bioculture'/'media_aida_ying_2025').is_dir()),None)
if ROOT is None: raise FileNotFoundError('data/bioculture/media_aida_ying_2025 경로 없음')
DATA=ROOT/'data'/'bioculture'/'media_aida_ying_2025'; R=ROOT/'results'/'bioculture'
MAPPING_REPORT=R/'B06_01_link_audit_v2'/'link_audit_report_v2.json'
B06_REPORT=R/'B06_02_growth_curves_v2'/'B06_02_validation_report.json'
B06_CURVES=R/'B06_02_growth_curves_v2'/'VALIDATED_curve_metrics_with_metadata.csv'
MEDIUM=DATA/'BW25113_Medium composition.xlsx'
OUT=R/'B07_00_ml_data_audit_v2'; OUT.mkdir(parents=True,exist_ok=True)
for f in (MAPPING_REPORT,B06_REPORT,B06_CURVES,MEDIUM):
    if not f.exists(): raise FileNotFoundError(f)
a=json.loads(MAPPING_REPORT.read_text(encoding='utf-8'))
b=json.loads(B06_REPORT.read_text(encoding='utf-8'))
assert a.get('join_validated') is True
assert b.get('qc_flags_persisted_and_verified') is True
print('Project:',ROOT)

## 2. 실제 열 이름 및 ID 구조 검사

식별자 `Label`, `Assay ID`, `Condition ID`는 항상 **문자열**로 유지합니다. Excel 성분 열의 괄호 단위는 그대로 보존하며 자동 환산하지 않습니다.

In [ ]:
IDS=['Label','Assay ID','Condition ID']
dtypes={x:'string' for x in IDS}
raw=pd.read_excel(MEDIUM,engine='openpyxl',dtype=dtypes)
curves=pd.read_csv(B06_CURVES,encoding='utf-8-sig',dtype=dtypes,low_memory=False)
assert set(IDS).issubset(raw.columns) and set(IDS).issubset(curves.columns)
assert not raw[IDS].isna().any().any() and not curves[IDS].isna().any().any()
assert not raw.duplicated(IDS).any() and not curves.duplicated(['round','Label']).any()
assert len(raw)==len(curves)==13608, (len(raw),len(curves))
print('Medium:',raw.shape,'Curves:',curves.shape)
print('Medium columns:',list(raw.columns))
print('B06 quality flags:',[c for c in curves if c.startswith('flag_')])
print('Conditions:',raw['Condition ID'].nunique(),curves['Condition ID'].nunique())


## 3. 무손실 결합, 실제 불일치 감사 및 조성 식별자 생성

모든 성분 열을 사용하여 수치값의 결정적 문자열 직렬화 후 SHA256으로 `medium_profile_id`를 만든다. `NaN`은 별도 마커로 표현하며 **결측을 0으로 채우지 않는다.** 원래 `Condition ID`와 별도로 추적·검증한다. 두 개 이상 서로 다른 조성을 포함한 원본 ID는 감사 CSV에 모두 기록한다. 데이터는 자동 수정하거나 행을 삭제하지 않는다.

In [ ]:
# 실제 원본의 불일치 여부는 아래 자동 감사에서 기록합니다.
# v1에서 삽입한 진단 셀은 여기서는 제거하여 실행 순서 의존성을 없앱니다.
print('Next cell: full composition and ID audit (no manual correction).')

In [ ]:
feature_cols=[c for c in raw.columns if c not in IDS]
assert feature_cols
joined=curves.merge(raw[IDS+feature_cols],on=IDS,how='left',validate='one_to_one',indicator=True)
assert joined['_merge'].eq('both').all() and len(joined)==len(curves)
joined=joined.drop(columns='_merge')
non_numeric={c:int(pd.to_numeric(raw[c],errors='coerce').isna().sum()-raw[c].isna().sum()) for c in feature_cols}
non_numeric={k:v for k,v in non_numeric.items() if v>0}
assert not non_numeric, f'농도 비수치 셀: {non_numeric}'
for c in feature_cols: joined[c]=pd.to_numeric(joined[c],errors='coerce')
# 원본에 없는 조성을 만들어내지 않고 동일한 수치 조합에 동일한 ID 부여.
# IEEE float64의 17 유효 숫자 표기로 표현하고 열 이름도 서명에 포함.
def canonical_num(x):
    if pd.isna(x): return '<MISSING>'
    v=float(x)
    if not np.isfinite(v): raise ValueError('비유한 배지 농도')
    if v==0: v=0.0
    return format(v,'.17g')
def profile_id(values):
    payload=json.dumps([(col,canonical_num(val)) for col,val in zip(feature_cols,values)],ensure_ascii=False,separators=(',',':'))
    return 'MP_'+hashlib.sha256(payload.encode('utf-8')).hexdigest()[:20]
profiles=raw[feature_cols].apply(pd.to_numeric,errors='raise')
# 조성 패턴이 반복되므로 중복을 먼저 줄여 대량 행 단위 해시 작업을 최소화.
unique_profiles=profiles.drop_duplicates().copy()
unique_profiles['medium_profile_id']=[profile_id(r) for r in unique_profiles[feature_cols].itertuples(index=False,name=None)]
assert unique_profiles.medium_profile_id.is_unique
mapping=profiles.merge(unique_profiles,on=feature_cols,how='left',validate='many_to_one',sort=False)
assert len(mapping)==len(raw) and mapping.medium_profile_id.notna().all()
raw_profile=raw[IDS].copy();raw_profile['medium_profile_id']=mapping.medium_profile_id.to_numpy()
joined=joined.merge(raw_profile,on=IDS,how='left',validate='one_to_one')
assert joined.medium_profile_id.notna().all() and len(joined)==13608
# 원본 ID가 가리키는 조성 수 조사.
id_profiles=joined.groupby('Condition ID')['medium_profile_id'].nunique()
conflict_ids=id_profiles[id_profiles>1].index.tolist()
conflict_detail=(joined.loc[joined['Condition ID'].isin(conflict_ids),IDS+['round','medium_profile_id']+feature_cols]
    .sort_values(['Condition ID','Assay ID']))
conflict_detail.to_csv(OUT/'original_condition_id_conflicts.csv',index=False,encoding='utf-8-sig')
conflict_summary=(joined.groupby(['Condition ID','medium_profile_id'],as_index=False)
                  .agg(n_curves=('Label','size'),n_rounds=('round','nunique')))
conflict_summary=conflict_summary.loc[conflict_summary['Condition ID'].isin(conflict_ids)]
conflict_summary.to_csv(OUT/'original_condition_variants.csv',index=False,encoding='utf-8-sig')
# 전역 조성 ID와 원본 Condition ID가 서로 다를 수 있음도 확인.
profile_to_ids=joined.groupby('medium_profile_id')['Condition ID'].nunique()
print('Original IDs:',joined['Condition ID'].nunique())
print('Distinct compositions:',joined.medium_profile_id.nunique())
print('Original ID conflicts:',conflict_ids)
print('Same composition under multiple original IDs:',int((profile_to_ids>1).sum()))
print(conflict_summary.to_string(index=False))
assert set(conflict_ids)=={'Cond00577','Cond00578'}, '예상과 다른 새로운 Condition ID 충돌: 원본 확인 필요'

## 4. 입력 특성 감사: 단위, 누락, 상수 및 변동성

단위가 열 이름에 나타나더라도 자동으로 변환하거나 정규화하지 않습니다. 빈 값은 무조건 0으로 대체하지 않습니다.

In [ ]:
audit=[]
for col in feature_cols:
    s=pd.to_numeric(joined[col],errors='coerce')
    audit.append({'feature':col,'missing_rows':int(s.isna().sum()),'missing_frac':float(s.isna().mean()),
                  'unique_nonmissing':int(s.nunique(dropna=True)),'minimum':float(s.min()) if s.notna().any() else None,
                  'maximum':float(s.max()) if s.notna().any() else None,'negative_count':int((s<0).sum())})
feat_audit=pd.DataFrame(audit)
feat_audit['constant_or_empty']=feat_audit['unique_nonmissing']<=1
feat_audit.to_csv(OUT/'medium_feature_audit.csv',index=False,encoding='utf-8-sig')
print('Composition fields:',len(feature_cols),'constant/empty:',int(feat_audit.constant_or_empty.sum()),
      'fields with missing:',int((feat_audit.missing_rows>0).sum()))
display(feat_audit.head(15))


## 5. 예측 목표 후보 및 품질 플래그 감사

원본 평가 `K`, `r`과 B06 탐색적 관측 지표를 **서로 다른 후보 목표**로 관리합니다. B06-03의 로지스틱 결과는 840개 선택 곡선에만 부분적으로 존재하므로, 현재 주요 지도학습 레이블로 채택하지 않습니다.

In [ ]:
targets=[c for c in ('K','r','observed_max','growth_log_slope_per_h') if c in joined.columns]
assert 'K' in targets and 'r' in targets
quality=[c for c in joined if c.startswith('flag_')]
target_audit=pd.DataFrame([{'target':c,'available':int(pd.to_numeric(joined[c],errors='coerce').notna().sum()),
                            'missing':int(pd.to_numeric(joined[c],errors='coerce').isna().sum()),
                            'meaning':'원본 평가표' if c in ('K','r') else 'B06 탐색적 계산'} for c in targets])
target_audit.to_csv(OUT/'target_candidate_audit.csv',index=False,encoding='utf-8-sig')
print(target_audit.to_string(index=False))
print('Quality flag columns:',quality)
print('Basic model candidates:',int(joined['flag_model_candidate'].sum()))


## 6. 원본 Condition ID와 조성 ID 기준 반복 구조를 모두 기록

원본 조건 식별자에 의한 요약은 감사 목적으로만 사용한다. **ML 조건 수준의 단위는 `medium_profile_id`**로 삼고, 원본 Condition ID가 여러 조성에 걸쳐 있는 경우 이를 숨기지 않는다.

In [ ]:
cond_original=joined.groupby('Condition ID',dropna=False).agg(n_curves=('Label','size'),
    n_profiles=('medium_profile_id','nunique'),n_rounds=('round','nunique')).reset_index()
cond_original.to_csv(OUT/'original_condition_repetition_audit.csv',index=False,encoding='utf-8-sig')
cond=joined.groupby('medium_profile_id',dropna=False).agg(n_curves=('Label','size'),
    n_original_conditions=('Condition ID','nunique'),n_rounds=('round','nunique'),
    n_qc_candidate=('flag_model_candidate','sum'),n_K=('K','count'),n_r=('r','count'),
    K_median=('K','median'),r_median=('r','median')).reset_index()
cond.to_csv(OUT/'medium_profile_repetition_audit.csv',index=False,encoding='utf-8-sig')
round_condition=joined.groupby(['round','medium_profile_id'],as_index=False).agg(n_curves=('Label','size'),n_K=('K','count'),n_r=('r','count'))
round_condition.to_csv(OUT/'round_medium_profile_counts.csv',index=False,encoding='utf-8-sig')
print('Original IDs:',len(cond_original),'Distinct medium profiles:',len(cond))
print('Number of curves:',len(joined),'Median curves/profile:',cond.n_curves.median())
display(cond.head())

## 7. 데이터 누출 방지 계획

동일 조성은 서로 다른 원본 Condition ID에서 등장하더라도 하나의 그룹에 둔다. 최종 일반화 평가는 **조성 그룹 기준의 분리**를 사용한다. 라운드 간 일반화 검증은 별개 질문이다. 이 단계에서 학습·분할을 수행하지 않는다.

In [ ]:
groups=joined['medium_profile_id'].astype(str)
assert groups.nunique()==len(cond)
unique_profile=joined[['medium_profile_id']+feature_cols].drop_duplicates()
assert len(unique_profile)==groups.nunique()
assert not unique_profile.duplicated('medium_profile_id').any()
# 만일 한 프로필이 두 다른 조성을 가리킨다면 치명적인 충돌.
profile_check=joined.groupby('medium_profile_id')[feature_cols].nunique(dropna=False)
assert (profile_check<=1).all().all()
quality=[c for c in joined if c.startswith('flag_')]
leakage_plan={
    'evaluation_group':'medium_profile_id (exact complete composition, not raw Condition ID)',
    'original_id_preserved':True,
    'split_rule':'group-disjoint train/test and group-aware CV by medium_profile_id',
    'must_not_use_as_predictors':['Label','Assay ID','Condition ID','medium_profile_id','round','K','r']+quality,
    'allowed_predictors':'audited numeric medium components only; preprocessing fit on training folds',
    'warning':'random curve-level split is invalid for unseen-composition generalization',
    'additional_validation':'hold-out by round is a separate distribution-shift question',
    'model_training_performed':False}
(OUT/'leakage_prevention_plan.json').write_text(json.dumps(leakage_plan,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps(leakage_plan,ensure_ascii=False,indent=2))

## 8. 곡선별·조성별 감사 테이블 저장 및 재읽기 검증

원본 `Condition ID`는 곡선별 표에서 보존되며, 조성별 테이블은 전역 `medium_profile_id`를 유일키로 사용한다. 원본 조건 ID를 강제로 단일 조성에 합치거나 평균내지 않는다.

In [ ]:
curve_path=OUT/'ml_curve_level_audit_table.csv'; condition_path=OUT/'ml_medium_profile_level_audit_table.csv'
joined.to_csv(curve_path,index=False,encoding='utf-8-sig')
condition_table=unique_profile.merge(cond,on='medium_profile_id',validate='one_to_one')
condition_table.to_csv(condition_path,index=False,encoding='utf-8-sig')
curve_disk=pd.read_csv(curve_path,encoding='utf-8-sig',dtype={x:'string' for x in IDS},low_memory=False)
condition_disk=pd.read_csv(condition_path,encoding='utf-8-sig')
assert len(curve_disk)==13608 and len(condition_disk)==len(cond)
assert not curve_disk.duplicated(['round','Label']).any()
assert not condition_disk.duplicated('medium_profile_id').any()
assert set(curve_disk['medium_profile_id'])==set(condition_disk['medium_profile_id'])
assert curve_disk.groupby('medium_profile_id').size().sum()==13608
print('Disk reload passed:',curve_disk.shape,condition_disk.shape)

## 9. 검증 기록 및 산출물 해시

In [ ]:
def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''): h.update(chunk)
    return h.hexdigest()
report={'status':'COMPOSITION_AUDIT_PASSED_NOT_MODEL_VALIDATION',
        'n_curves':int(len(joined)), 'n_original_condition_ids':int(joined['Condition ID'].nunique()),
        'n_distinct_medium_profiles':int(len(cond)),
        'original_ids_with_multiple_compositions':conflict_ids,
        'n_original_id_conflicts':len(conflict_ids),
        'n_medium_features':len(feature_cols),'n_constant_or_empty_features':int(feat_audit.constant_or_empty.sum()),
        'n_model_candidates_B06':int(joined.flag_model_candidate.sum()),
        'targets_candidates':targets,'composition_consistent_by_medium_profile':True,
        'data_split_executed':False,'model_fitted':False,
        'caveat':'Source ID anomaly is documented, not corrected. K/r definitions, units and replicate independence not proven.'}
(OUT/'B07_00_validation_report.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
inputs=[MEDIUM,MAPPING_REPORT,B06_REPORT,B06_CURVES]
manifest=[{'kind':'input','path':str(p),'sha256':sha256(p),'bytes':p.stat().st_size} for p in inputs]
manifest +=[{'kind':'output','path':str(p),'sha256':sha256(p),'bytes':p.stat().st_size} for p in sorted(OUT.iterdir()) if p.is_file() and p.name!='file_manifest_sha256.csv']
pd.DataFrame(manifest).to_csv(OUT/'file_manifest_sha256.csv',index=False,encoding='utf-8-sig')
print(json.dumps(report,ensure_ascii=False,indent=2))

## 연습문제 (정답은 별도의 Word 해설서에 수록)

1. 하나의 `Condition ID` 아래 두 조성이 있을 때 조건별 중앙값을 계산하면 어떤 문제가 생기는가?
2. `Condition ID` 대신 `medium_profile_id`로 그룹 분할할 때 방지되는 데이터 누출은 무엇인가?
3. 동일 조성이 여러 원본 Condition ID 아래 존재할 수 있다. 이를 어떻게 점검할 수 있는가?
4. 배지 성분의 결측을 0으로 대체하지 않고 처리해야 하는 이유는 무엇인가?
5. 원본 `K`/`r`와 B06 탐색적 목표 지표를 구별해야 하는 이유를 설명하라.

**다음 단계:** v2 실행 결과에서 원본 충돌과 새 조성 그룹 수를 검증한 뒤 B07-01 탐색적 통계 분석으로 진행한다.